# Synthetic Tweet Generation — Multi-Model Sweep

Generates synthetic data from **multiple HuggingFace open-weight LLMs** in a single run on Colab A100. Each model uses the same prompt grid, same `TWEETS_PER_CELL`, same `SEED_BASE` as `01b_DataGenerator_Llama.ipynb` — outputs are directly comparable.

**Workflow:**
1. Pilot run (`TWEETS_PER_CELL=50`) across the model list → audit each, eyeball samples
2. Pick the 2–3 most-divergent generators (your bias-transfer signal)
3. Full run (`TWEETS_PER_CELL=1200`) on the selected ones only → feed to classifier training

**Why this is cheap:** weight download dominates wall-clock; once a model is in VRAM, generation at batch=32 on A100 is ~80–150 tokens/sec/sample. 50/cell × 6 cells = ~300 calls = ~3–5 min/model after load. Six models in a pilot = ~45 min wall clock + 6 × ~3 min downloads.

**Memory hygiene:** between models the previous weights are explicitly freed (`del model; torch.cuda.empty_cache()`). Each model is loaded fresh; no two models in VRAM simultaneously.

**Project layout (Drive):**
```
MyDrive/PoliticalBiasProject/
  data/synthetic_data/
    {model-tag}_synthetic_{N}per_cell_{ts}.csv     ← one CSV per model
```

## 0. Colab setup

In [ ]:
!pip install -q transformers==4.45.0 accelerate==0.34.0 huggingface_hub pandas tqdm

from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/PoliticalBiasProject'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)
print('Working dir:', os.getcwd())

## 1. HF login + GPU check

In [ ]:
import os, re, gc, time, hashlib
from datetime import datetime
from pathlib import Path

import pandas as pd
import torch
from tqdm.auto import tqdm
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer

HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    pass
if not HF_TOKEN:
    HF_TOKEN = os.environ.get('HF_TOKEN')

if HF_TOKEN:
    login(token=HF_TOKEN)
    print('HF login OK')
else:
    print('No HF_TOKEN — set it in Colab Secrets (key icon) before loading gated models.')

print('CUDA :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU  :', torch.cuda.get_device_name(0))
    print('VRAM :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

## 2. Run configuration

**Edit `MODEL_LIST` to add/remove generators.** Comment out a line to skip. All listed models must:
- Be available on HuggingFace (gated models require token approval first)
- Fit on A100 40GB in bf16 (≤14B params is safe; larger needs 4-bit quantization — different code path)
- Have a chat template compatible with `tokenizer.apply_chat_template`

Suggested defaults below cover four alignment regimes:

In [ ]:
MODEL_LIST = [
    # "meta-llama/Llama-3.1-8B-Instruct",     # ALREADY DONE (commented out to avoid re-running)
    "mistralai/Mistral-7B-Instruct-v0.3",   # Mistral, lighter alignment - direct comparison to Wagner et al. 2025
    "google/gemma-2-9b-it",                 # Google, RLHF distinct from Meta/Mistral
    "Qwen/Qwen2.5-7B-Instruct",             # Alibaba, different RLHF philosophy
    "meta-llama/Llama-3.2-3B-Instruct",     # Same Meta family, smaller - size-scaling probe
    # "microsoft/Phi-3.5-mini-instruct",    # Microsoft, small
    # "tiiuae/falcon3-7b-instruct",         # TII, lighter alignment
]

TWEETS_PER_CELL = 50      # 50 = pilot (recommended for first sweep), 1200 = full
SEED_BASE       = 42
BATCH_SIZE      = 32

TARGETS = ["Donald Trump", "Joe Biden", "Bernie Sanders"]
STANCES = ["FAVOR", "AGAINST"]

# Absolute path to the Drive project — guarantees CSVs land on Drive even if chdir didn't run
PROJECT_DIR = Path("/content/drive/MyDrive/PoliticalBiasProject")
OUTPUT_DIR  = PROJECT_DIR / "data" / "synthetic_data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output dir      : {OUTPUT_DIR}")

print(f"Models to run   : {len(MODEL_LIST)}")
for i, m in enumerate(MODEL_LIST, 1):
    print(f"  {i}. {m}")
print(f"Tweets per cell : {TWEETS_PER_CELL}")
print(f"Total cells     : {len(TARGETS) * len(STANCES)}")


## 3. Prompt grid (identical to `01_DataGenerator.ipynb`)

In [ ]:
SYSTEM_PROMPT = (
    'You are a creative-writing assistant generating fictional social-media '
    'posts for an academic NLP research dataset. The posts are training data '
    'for a stance-classification model — they are not real opinions and will '
    'not be published as real tweets. '
    'Real tweets are short (often 80–160 characters), informal, conversational, '
    'and may include hashtags or casual phrasing. '
    'Output only the post text. No quotes, no preamble, no caveats, no disclaimers.'
)

TEMPLATES = [
    'Write a single tweet that expresses a stance {direction} {target}, focused on {topic}. Format it as {fmt}. Stay under 200 characters.',
    'Compose one tweet from a regular Twitter user whose stance is {direction} {target}. The tweet is about {topic} and reads like {fmt}. Maximum 200 characters.',
    'Generate a realistic tweet expressing a position {direction} {target}. The subject is {topic} and the tone should fit {fmt}. Keep under 200 characters.',
    'Produce a single short tweet — stance: {direction} {target}, topic: {topic}, format: {fmt}. Real Twitter user voice. Under 200 characters.',
]
TOPICS = [
    'their record in office or public life',
    'their position on a policy issue',
    'the campaign trail',
    'a recent news cycle moment',
    'the primary race',
    'the upcoming general election',
]
FORMATS = [
    'a brief personal reaction',
    'a one-line observation',
    'a rhetorical question',
    'a sharp one-liner',
]

def build_prompt(target, stance, idx):
    direction = 'in favor of' if stance == 'FAVOR' else 'against'
    t  = TEMPLATES[idx % len(TEMPLATES)]
    tp = TOPICS[(idx // len(TEMPLATES)) % len(TOPICS)]
    fm = FORMATS[(idx // (len(TEMPLATES) * len(TOPICS))) % len(FORMATS)]
    return t.format(direction=direction, target=target, topic=tp, fmt=fm)

def build_chat(target, stance, idx):
    return [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': build_prompt(target, stance, idx)},
    ]

REFUSAL_HEADS = ("i can't", "i cannot", "i'm sorry", 'as an ai',
                 "i won't", "i'm not able", "i'm unable")

def is_refusal(text):
    if not text or len(text) < 20:
        return True
    return text.lower().lstrip('"\' ').startswith(REFUSAL_HEADS)

def normalize(text):
    return re.sub(r'[^\w\s]', '', text.lower()).strip() if text else ''

for i in range(3):
    print(f'[{i}] {build_prompt("Donald Trump", "FAVOR", i)}')

## 4. Per-model generation function

Loads the model, runs the same dedup-and-retry loop, returns a DataFrame. Some models (Gemma) reject `system` role — the function falls back to merging system text into the user message.

**Models that don't accept `system` role:** Gemma family. Detected at runtime; falls back automatically.

In [ ]:
def chat_messages(target, stance, idx, supports_system=True):
    if supports_system:
        return [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': build_prompt(target, stance, idx)},
        ]
    # Gemma-style: system content prepended to user message
    return [
        {'role': 'user', 'content': SYSTEM_PROMPT + '\n\n' + build_prompt(target, stance, idx)},
    ]

def detect_supports_system(tokenizer):
    """Probe whether the chat template accepts a system role."""
    try:
        tokenizer.apply_chat_template(
            [{'role': 'system', 'content': 'x'}, {'role': 'user', 'content': 'y'}],
            add_generation_prompt=True, tokenize=False,
        )
        return True
    except Exception:
        return False

def run_one_model(model_id, tweets_per_cell, batch_size=BATCH_SIZE):
    print(f'\n{"=" * 60}\n  {model_id}\n{"=" * 60}')
    t_load = time.time()

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = 'left'

    model = AutoModelForCausalLM.from_pretrained(
        model_id, torch_dtype=torch.bfloat16, device_map='auto',
    )
    model.eval()
    supports_system = detect_supports_system(tokenizer)
    print(f'  Loaded in {time.time()-t_load:.1f}s | system role: {supports_system}')

    # Smoke test
    msgs = chat_messages('Donald Trump', 'FAVOR', 0, supports_system)
    inputs = tokenizer.apply_chat_template(
        msgs, add_generation_prompt=True, return_tensors='pt', return_dict=True,
    ).to(model.device)
    torch.manual_seed(SEED_BASE)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=70, do_sample=True,
            temperature=0.9, top_p=0.95, pad_token_id=tokenizer.pad_token_id,
        )
    smoke = tokenizer.decode(out[0, inputs['input_ids'].shape[-1]:], skip_special_tokens=True).strip()
    print(f'  Smoke test: {smoke[:140]}...')

    # Batched generation
    @torch.no_grad()
    def generate_batch(chat_list, seed):
        torch.manual_seed(seed)
        inputs = tokenizer.apply_chat_template(
            chat_list, add_generation_prompt=True, return_tensors='pt',
            return_dict=True, padding=True,
        ).to(model.device)
        prompt_len = inputs['input_ids'].shape[-1]
        out = model.generate(
            **inputs, max_new_tokens=70, do_sample=True,
            temperature=0.9, top_p=0.95, pad_token_id=tokenizer.pad_token_id,
        )
        texts = tokenizer.batch_decode(out[:, prompt_len:], skip_special_tokens=True)
        return [t.strip().strip('"\'') for t in texts]

    rows = []
    t_gen = time.time()
    for ti, target in enumerate(TARGETS):
        for si, stance in enumerate(STANCES):
            offset    = (ti * len(STANCES) + si) * 100_000
            seen_norm = set()
            kept      = 0
            idx       = 0
            max_tries = tweets_per_cell * 2

            pbar = tqdm(total=tweets_per_cell, desc=f'{target}/{stance}', leave=False)
            while kept < tweets_per_cell and idx < max_tries:
                this_batch = min(batch_size, max_tries - idx)
                chats = [chat_messages(target, stance, idx + j, supports_system) for j in range(this_batch)]
                seeds = [SEED_BASE + offset + idx + j for j in range(this_batch)]
                try:
                    outputs = generate_batch(chats, seed=seeds[0])
                    err = None
                except Exception as e:
                    outputs, err = [None] * this_batch, str(e)
                    print(f'  [ERROR] {err[:200]}')
                    break
                for j, (text, seed) in enumerate(zip(outputs, seeds)):
                    refused = is_refusal(text)
                    norm    = normalize(text) if text else ''
                    if not refused and not err and norm and norm not in seen_norm:
                        seen_norm.add(norm)
                        rows.append({
                            'Tweet': text, 'Target': target, 'Stance': stance,
                            'model': model_id, 'prompt_idx': idx + j, 'seed': seed,
                            'refused': False, 'error': None,
                        })
                        kept += 1
                        pbar.update(1)
                        if kept >= tweets_per_cell:
                            break
                idx += this_batch
            pbar.close()
            print(f'  {target}/{stance}: kept {kept}/{tweets_per_cell} after {idx} tries')
    print(f'  Generation in {time.time()-t_gen:.1f}s')

    # Free VRAM before next model
    del model
    del tokenizer
    gc.collect()
    torch.cuda.empty_cache()

    # Clear HF disk cache for this model so the next download has room
    import shutil
    cache_root = Path.home() / ".cache" / "huggingface" / "hub"
    model_cache_name = "models--" + model_id.replace("/", "--")
    model_cache_path = cache_root / model_cache_name
    if model_cache_path.exists():
        shutil.rmtree(model_cache_path)
        print(f"  Freed disk cache: {model_cache_path}")

    return pd.DataFrame(rows)

## 5. Run all models

In [ ]:
results = {}    # model_id -> DataFrame
csv_paths = {}  # model_id -> Path

for model_id in MODEL_LIST:
    try:
        df = run_one_model(model_id, tweets_per_cell=TWEETS_PER_CELL)
    except Exception as e:
        print(f'\n  [SKIPPED] {model_id} — {str(e)[:200]}')
        continue

    ts  = datetime.now().strftime('%Y%m%d_%H%M%S')
    tag = model_id.replace('/', '_')
    path = OUTPUT_DIR / f'{tag}_synthetic_{TWEETS_PER_CELL}per_cell_{ts}.csv'
    df.to_csv(path, index=False)
    print(f'  ✅ Wrote {len(df)} rows → {path}')

    results[model_id] = df
    csv_paths[model_id] = path

print(f'\n{"=" * 60}')
print(f'  Done. {len(results)}/{len(MODEL_LIST)} models successful.')
print(f'{"=" * 60}')

## 6. Quick comparison — pick which models to scale up

Headline metrics per model: char length, hashtag rate, top-trigram concentration. Use this to decide which 2–3 models to run at full 1200/cell scale.

**Decision rules:**
- Most-divergent length / hashtag profile → strongest alignment-regime contrast
- High top-trigram concentration → heavy templating fingerprint (interesting bias signal)
- Low refusal rate, high uniqueness → reliable for downstream training

In [ ]:
import re as _re
from collections import Counter

def quick_summary(df):
    df = df.copy()
    df['char_len']   = df['Tweet'].str.len()
    df['n_hashtags'] = df['Tweet'].str.count(r'#\w+')
    df['_norm']      = df['Tweet'].apply(normalize)

    # Top-trigram concentration: % of tweets containing the most-common trigram in their cell
    top_tri_pct = []
    for (t, s), grp in df.groupby(['Target', 'Stance']):
        c = Counter()
        for tw in grp['Tweet']:
            words = _re.findall(r'[a-z]+', str(tw).lower())
            for i in range(len(words) - 2):
                c[(words[i], words[i+1], words[i+2])] += 1
        if c:
            top_tri_pct.append(c.most_common(1)[0][1] / len(grp))

    return {
        'n':                    len(df),
        'char_len':             round(df['char_len'].mean(), 1),
        'n_hashtags':           round(df['n_hashtags'].mean(), 2),
        'unique_pct':           round(df['_norm'].nunique() / len(df) * 100, 1),
        'top_trigram_pct_avg':  round(sum(top_tri_pct)/len(top_tri_pct) * 100, 1) if top_tri_pct else 0.0,
    }

rows = []
for model_id, df in results.items():
    s = quick_summary(df)
    s['model'] = model_id.split('/')[-1]
    rows.append(s)

leaderboard = pd.DataFrame(rows).set_index('model')[
    ['n', 'char_len', 'n_hashtags', 'unique_pct', 'top_trigram_pct_avg']
]
print('Quick-comparison leaderboard:')
print(leaderboard)

# Sample a few tweets per model for eyeball comparison
print('\n\n3 samples per model (Trump-FAVOR):')
for model_id, df in results.items():
    print(f'\n--- {model_id} ---')
    sub = df[(df.Target == 'Donald Trump') & (df.Stance == 'FAVOR')]
    for tw in sub['Tweet'].head(3):
        print(f'  • {tw[:180]}')

## 7. Next steps

1. **Eyeball the samples and the leaderboard.** Note which models look most distinct from each other in length / hashtag profile / lexical voice.
2. **Sync CSVs back to local repo:** all `data/synthetic_data/*_synthetic_50per_cell_*.csv` files for the audit.
3. **Run the full audit** ([03_SyntheticAudit.ipynb](03_SyntheticAudit.ipynb)) — add the new CSV paths to the `SYN_PATHS` dict in cell 2; everything else iterates automatically.
4. **Pick 2–3 models to scale up.** Strongest paper story: pick the most distinctive ones, not the best-quality ones.
5. **Set `TWEETS_PER_CELL = 1200`** in cell 2 above, comment out the models you're skipping in `MODEL_LIST`, rerun. ~30 min/model on A100.
6. **Train RoBERTa classifiers** ([04_RoBERTa_FineTune.ipynb](04_RoBERTa_FineTune.ipynb)) on the selected synthetic datasets.